In [18]:
import pandas as pd

# 1. 파일 불러오기 (시트명: 사용자 발화)
file_path = '웰니스 대화 스크립트.xlsx'
df = pd.read_excel(file_path, sheet_name='사용자 발화')

# 2. 인텐트 빈칸 채우기 (ffill)
# 인텐트가 생략된 행들에 대해 위의 값을 그대로 채워넣습니다.
df['intent'] = df['intent'].ffill()

# 3. 'utterance(2차) ' 열만 선택하여 데이터 프레임 구성
# 컬럼명 끝에 공백이 있을 수 있으므로 정확히 매칭합니다.
# 만약 오류가 난다면 'utterance(2차)'로 수정해 보세요.
wellness_df = df[['utterance(2차) ', 'intent']].copy()

# 4. 발화 내용(utterance(2차))이 없는 행 제거
wellness_df = wellness_df.dropna(subset=['utterance(2차) '])

# 5. 논문 기준 19가지 우울 관련 인텐트 필터링
target_intents = [
    '정신증상/우울감', '정신증상/슬픔', '정신증상/외로움', '정신증상/분노', '정신증상/무기력', 
    '정신증상/감정조절 이상', '정신증상/상실감', '정신증상/식욕 저하', '정신증상/식욕 증가', 
    '정신증상/불면', '정신증상/초조함', '정신증상/피로', '정신증상/죄책감', '정신증상/집중력저하', 
    '정신증상/자신감 저하', '정신증상/자존감 저하', '정신증상/절망감', '정신증상/자살 충동', '정신증상/불안'
]

# 공백 차이로 인한 누락 방지 로직
target_no_space = [i.replace(' ', '') for i in target_intents]
wellness_df['intent_no_space'] = wellness_df['intent'].str.replace(' ', '')

# 필터링 실행
wellness_final = wellness_df[wellness_df['intent_no_space'].isin(target_no_space)].copy()

# 6. 최종 인텐트 정제 (논문 표기법대로 '정신증상/' 제거)
wellness_final['intent'] = wellness_final['intent'].str.split('/').str[-1].str.strip()

# 7. 결과 정리 및 저장
# 컬럼명을 논문 형식인 utterance와 intent로 통일합니다.
wellness_final = wellness_final[['utterance(2차) ', 'intent']].rename(columns={'utterance(2차) ': 'utterance'})
wellness_final.to_csv('pre_wellness.csv', index=False, encoding='utf-8-sig')

print(f"✅ 웰니스 전처리 완료 (utterance(2차) 열 사용)")
print(f"추출된 데이터 수: {len(wellness_final)}건")
print("\n--- 인텐트별 데이터 분포 (일부) ---")
print(wellness_final['intent'].value_counts().head())

✅ 웰니스 전처리 완료 (utterance(2차) 열 사용)
추출된 데이터 수: 19677건

--- 인텐트별 데이터 분포 (일부) ---
intent
초조함    2697
불면     1867
슬픔     1789
우울감    1657
무기력    1288
Name: count, dtype: int64
